# Entrenar «Hola Fructus» (openWakeWord, en español)

Entrena el modelo que despierta a Fructus al oír «Hola Fructus». Es la **prueba de viabilidad** (actividad 4): el modelo que sale de
aquí se mide después **en la computadora del proyecto** con el ruido real del puesto (`medir.py`). Ese ruido no se sube nunca aquí.

**Qué subir:** solo un `grabaciones.zip` con «Hola Fructus» dichos por el equipo, con su permiso. Una carpeta por persona
(`grabaciones/dueno/…`, `grabaciones/ayudante1/…`).

**Licencia:** las características de entrenamiento (ACAV100M) son CC BY-NC-SA 4.0 (no comercial). El modelo sirve para la demo
universitaria, no para vender.

**Tiempo:** entre 1 y 2 horas en total. No hace falta GPU. Baja unos 19 GB al disco de Colab, no al tuyo.

Ejecuta las celdas en orden (Entorno de ejecución → Ejecutar todas). Generado por `scripts/voz/palabra-activacion/armar_cuaderno.py`:
no lo edites a mano.

## 1. Instalar

In [ ]:
!pip install -q "openwakeword==0.6.0" "piper-tts==1.8.0" torchinfo torchmetrics pronouncing audiomentations torch-audiomentations "speechbrain<1" acoustics mutagen "scipy>=1.14,<1.16" pyyaml soundfile onnx onnxscript pyarrow huggingface_hub

## 2. Los scripts del proyecto

In [ ]:
%%writefile compat_torchaudio.py
"""Vuelve a dar `torchaudio.load` y `torchaudio.info` con soundfile.

openWakeWord 0.6.0 (y speechbrain 0.5) leen audio con `torchaudio.info` y `torchaudio.load`. Desde torchaudio 2.9, `info` ya no
existe y `load` exige torchcodec: en un Colab de hoy el entrenamiento se cae al aumentar los clips. Este parche se aplica antes de
importar el entrenador. Solo hace falta para WAV/FLAC, que es lo que usa el entrenamiento.
"""

from types import SimpleNamespace

import soundfile as sf
import torch
import torchaudio


def _load(ruta, *args, **kwargs):
    datos, frecuencia = sf.read(str(ruta), dtype="float32", always_2d=True)
    return torch.from_numpy(datos.T.copy()), frecuencia


def _info(ruta, *args, **kwargs):
    i = sf.info(str(ruta))
    return SimpleNamespace(sample_rate=i.samplerate, num_frames=i.frames, num_channels=i.channels)


def aplicar() -> None:
    torchaudio.load = _load
    torchaudio.info = _info

In [ ]:
%%writefile preparar_datos.py
"""Baja lo que necesita el entrenamiento, sin librerías que hoy piden torchcodec (`datasets` y `torchaudio` nuevos).

    voces/       voces en español de Piper (España, México y Argentina)                     ~ 450 MB
    mit_rirs/    270 respuestas de sala del MIT, a 16 kHz                                    ~ 8 MB
    fondo/       ruido de fondo: clips de 10 s de AudioSet (bal_train), a 16 kHz mono       ~ 650 MB por fragmento
    openwakeword_features_ACAV100M_2000_hrs_16bit.npy  audio que no es la palabra           ~ 17.3 GB
    validation_set_features.npy                        para estimar falsas activaciones     ~ 185 MB

El cuaderno oficial de openWakeWord bajaba AudioSet en .tar, que ya no existe (ahora es parquet), y decodificaba con `datasets`.

Licencias: el código de openWakeWord es Apache 2.0. Las características ACAV100M son CC BY-NC-SA 4.0 (no comercial): un modelo
entrenado con ellas sirve para la demo universitaria, no para vender. Antes de producción se reentrena con audio propio.

Uso:
    python preparar_datos.py --destino . --fragmentos-fondo 2
    python preparar_datos.py --destino prueba --sin-caracteristicas --max-clips-fondo 20   # prueba chica
"""

import argparse
import io
import shutil
from pathlib import Path

import numpy as np
import soundfile as sf
from scipy.signal import resample_poly

FRECUENCIA = 16_000
VOCES = [
    "es/es_AR/daniela/high/es_AR-daniela-high",
    "es/es_ES/davefx/medium/es_ES-davefx-medium",
    "es/es_ES/sharvard/medium/es_ES-sharvard-medium",
    "es/es_ES/mls_10246/low/es_ES-mls_10246-low",
    "es/es_ES/mls_9972/low/es_ES-mls_9972-low",
    "es/es_MX/ald/medium/es_MX-ald-medium",
    "es/es_MX/claude/high/es_MX-claude-high",
]


def a_16k_mono(audio: np.ndarray, frecuencia: int) -> np.ndarray:
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if frecuencia != FRECUENCIA:
        mcd = np.gcd(frecuencia, FRECUENCIA)
        audio = resample_poly(audio, FRECUENCIA // mcd, frecuencia // mcd)
    return np.clip(audio * 32767, -32768, 32767).astype(np.int16)


def bajar_voces(destino: Path) -> None:
    from huggingface_hub import hf_hub_download

    destino.mkdir(parents=True, exist_ok=True)
    for v in VOCES:
        for ext in (".onnx", ".onnx.json"):
            ruta = hf_hub_download("rhasspy/piper-voices", v + ext, local_dir=destino / "_hf")
            Path(ruta).replace(destino / Path(v + ext).name)  # se mueve, no se copia: son ~70 MB cada una
    shutil.rmtree(destino / "_hf", ignore_errors=True)
    print(f"voces: {len(VOCES)} en {destino}")


def bajar_salas(destino: Path) -> None:
    from huggingface_hub import HfFileSystem

    fs = HfFileSystem()
    destino.mkdir(parents=True, exist_ok=True)
    archivos = fs.glob("datasets/davidscripka/MIT_environmental_impulse_responses/16khz/*.wav")
    for a in archivos:
        salida = destino / Path(a).name
        if not salida.exists():
            audio, frecuencia = sf.read(io.BytesIO(fs.cat_file(a)), dtype="float32")
            sf.write(salida, a_16k_mono(audio, frecuencia), FRECUENCIA, subtype="PCM_16")
    print(f"salas: {len(archivos)} en {destino}")


def bajar_fondo(destino: Path, fragmentos: int, max_clips: int | None) -> None:
    import pyarrow.parquet as pq
    from huggingface_hub import HfFileSystem

    fs = HfFileSystem()
    destino.mkdir(parents=True, exist_ok=True)
    hechos = 0
    for n in range(fragmentos):
        with fs.open(f"datasets/agkphysics/AudioSet/data/bal_train/{n:02d}.parquet", "rb") as f:
            pf = pq.ParquetFile(f)
            for g in range(pf.num_row_groups):  # por grupos: no carga 650 MB de una vez
                tabla = pf.read_row_group(g, columns=["video_id", "audio"])
                for vid, audio in zip(tabla.column("video_id").to_pylist(), tabla.column("audio").to_pylist()):
                    salida = destino / f"audioset_{vid}.wav"
                    if not salida.exists():
                        try:
                            datos, frecuencia = sf.read(io.BytesIO(audio["bytes"]), dtype="float32")
                        except RuntimeError:
                            continue  # algún clip viene dañado: se salta
                        sf.write(salida, a_16k_mono(datos, frecuencia), FRECUENCIA, subtype="PCM_16")
                    hechos += 1
                    if max_clips and hechos >= max_clips:
                        print(f"fondo: {hechos} clips en {destino}")
                        return
    print(f"fondo: {hechos} clips en {destino}")


def bajar_caracteristicas(destino: Path) -> None:
    from huggingface_hub import hf_hub_download

    for nombre in ("validation_set_features.npy", "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"):
        ruta = hf_hub_download("davidscripka/openwakeword_features", nombre, repo_type="dataset", local_dir=destino)
        print(f"{nombre}: {Path(ruta).stat().st_size / 2**30:.1f} GB")


def main() -> None:
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--destino", type=Path, default=Path("."))
    p.add_argument("--fragmentos-fondo", type=int, default=2, help="cada fragmento de AudioSet son ~500 clips de 10 s")
    p.add_argument("--max-clips-fondo", type=int, help="corta antes (para probar)")
    p.add_argument("--sin-caracteristicas", action="store_true", help="no baja los 17.5 GB (para probar)")
    a = p.parse_args()

    bajar_voces(a.destino / "voces")
    bajar_salas(a.destino / "mit_rirs")
    bajar_fondo(a.destino / "fondo", a.fragmentos_fondo, a.max_clips_fondo)
    if not a.sin_caracteristicas:
        bajar_caracteristicas(a.destino)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile generar_clips.py
"""Genera los clips de entrenamiento de «Hola Fructus» en español, donde los espera el entrenador de openWakeWord.

openWakeWord genera sus clips con una voz sintética en inglés. Aquí se generan con las voces en español de Piper (España, México
y Argentina), variando velocidad y entonación, y se suman las grabaciones reales del equipo. Deja:

    <salida>/positive_train  «Hola Fructus» para entrenar
    <salida>/positive_test   «Hola Fructus» para validar durante el entrenamiento
    <salida>/negative_train  frases parecidas o del puesto que NO deben despertarlo
    <salida>/negative_test

Todo a 16 kHz, mono, 16 bits. Las grabaciones reales de las personas en --reservar NO se usan aquí: quedan para medir
(medir.py), así el número final sale de voces que el modelo nunca oyó.

Uso:
    python generar_clips.py --voces voces/ --salida salida/hola_fructus --n-train 5000 --n-test 500 \
        --grabaciones grabaciones/ --reservar dueno
"""

import argparse
import random
import shutil
import subprocess
import tempfile
from pathlib import Path

import numpy as np
import soundfile as sf
from scipy.signal import resample_poly

FRECUENCIA = 16_000

POSITIVAS = ["Hola Fructus", "hola fructus", "Hola, Fructus", "¡Hola Fructus!", "Hola Fructus.", "Ola Fructus"]

# Lo que se dice en el puesto y suena parecido: no debe despertarlo. Incluye «Fructus» solo, para que haga falta el «hola».
NEGATIVAS = [
    "hola", "hola Lucho", "hola casera", "hola amigo", "hola señora", "hola, ¿cuánto está?", "hola, buenos días",
    "Fructus", "fruta", "frutas", "las frutas", "fructosa", "productos", "Fructuoso", "factura", "hazle factura",
    "ola", "hola Rosa", "hola don Lucho", "la fruta está buena", "tus frutas", "con fruta", "producto",
    "vendí diez cajas de mango a don Lucho", "se malograron tres jabas de papaya", "¿quién me debe?",
    "pagué ochenta de flete", "llegaron cincuenta cajas de plátano", "me yapeó", "lo debe", "cuánto vendí hoy",
    "¿cuántas cajas de mango me quedan?", "dos cajas de manzana", "fresa", "frutería", "fructífero", "hoy vendimos",
]


def a_16k(audio: np.ndarray, frecuencia: int) -> np.ndarray:
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if frecuencia != FRECUENCIA:
        mcd = np.gcd(frecuencia, FRECUENCIA)
        audio = resample_poly(audio, FRECUENCIA // mcd, frecuencia // mcd)
    pico = float(np.max(np.abs(audio))) or 1.0
    return (audio / pico * 0.9 * 32767).astype(np.int16)


LEIBLES = {".wav", ".flac", ".ogg"}
CONVERTIBLES = {".m4a", ".mp3", ".aac", ".mp4", ".opus", ".webm", ".caf"}  # lo que graba un celular


def leer(ruta: Path) -> tuple[np.ndarray, int]:
    """Audio como float32. Lo que graba un celular (M4A, MP3…) se convierte con ffmpeg, que Colab ya trae."""
    if ruta.suffix.lower() not in CONVERTIBLES:
        return sf.read(str(ruta), dtype="float32")
    if not shutil.which("ffmpeg"):
        raise SystemExit(f"Para leer {ruta.name} hace falta ffmpeg.")
    with tempfile.TemporaryDirectory() as tmp:
        wav = Path(tmp) / "a.wav"
        subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", str(ruta), "-ac", "1", "-ar", str(FRECUENCIA), str(wav)], check=True)
        return sf.read(str(wav), dtype="float32")


def cargar_voces(carpeta: Path, espeak: Path | None):
    from piper import PiperVoice

    voces = []
    for modelo in sorted(carpeta.glob("*.onnx")):
        # espeak no acepta rutas largas (unos 160 caracteres): si la del paquete lo es, se copia a una corta y se pasa aquí.
        voz = PiperVoice.load(str(modelo), **({"espeak_data_dir": espeak} if espeak else {}))
        voces.append((modelo.stem, voz, max(1, voz.config.num_speakers)))
    if not voces:
        raise SystemExit(f"No hay voces .onnx de Piper en {carpeta}")
    return voces


def sintetizar(voz, hablantes: int, texto: str, azar: random.Random) -> np.ndarray:
    from piper import SynthesisConfig

    config = SynthesisConfig(
        speaker_id=azar.randrange(hablantes) if hablantes > 1 else None,
        length_scale=azar.uniform(0.8, 1.35),  # más rápido o más lento, como se habla apurado o con calma
        noise_scale=azar.uniform(0.45, 0.9),
        noise_w_scale=azar.uniform(0.6, 1.0),
    )
    trozos = list(voz.synthesize(texto, syn_config=config))
    audio = np.concatenate([t.audio_int16_array for t in trozos]).astype(np.float32) / 32768
    return a_16k(audio, trozos[0].sample_rate)


def llenar(carpeta: Path, cuantos: int, textos: list[str], voces, azar: random.Random) -> None:
    carpeta.mkdir(parents=True, exist_ok=True)
    ya = len(list(carpeta.glob("*.wav")))
    for n in range(ya, cuantos):
        nombre, voz, hablantes = voces[n % len(voces)]
        audio = sintetizar(voz, hablantes, azar.choice(textos), azar)
        sf.write(carpeta / f"tts_{nombre}_{n:05d}.wav", audio, FRECUENCIA, subtype="PCM_16")
    print(f"{carpeta.name}: {cuantos} clips ({cuantos - ya} nuevos)")


def sumar_grabaciones(origen: Path, entrenar: Path, validar: Path, reservar: set[str], repetir: int, azar: random.Random) -> None:
    """Grabaciones reales en <origen>/<persona>/*.wav (o .m4a/.mp3 convertidos antes). Las reservadas no se tocan."""
    personas = [p for p in sorted(origen.iterdir()) if p.is_dir()]
    usadas = 0
    for persona in personas:
        if persona.name in reservar:
            print(f"grabaciones de «{persona.name}»: reservadas para medir, no se usan al entrenar")
            continue
        archivos = sorted(f for f in persona.iterdir() if f.suffix.lower() in LEIBLES | CONVERTIBLES)
        azar.shuffle(archivos)
        corte = max(1, len(archivos) // 10)  # una de cada diez, a validar durante el entrenamiento
        for i, archivo in enumerate(archivos):
            audio, frecuencia = leer(archivo)
            clip = a_16k(audio, frecuencia)
            destino, veces = (validar, 1) if i < corte else (entrenar, repetir)
            for r in range(veces):  # pesan más que los sintéticos: son las voces y el micrófono de verdad
                sf.write(destino / f"real_{persona.name}_{archivo.stem}_{r}.wav", clip, FRECUENCIA, subtype="PCM_16")
            usadas += 1
    print(f"grabaciones reales usadas: {usadas}")


def main() -> None:
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--voces", type=Path, required=True, help="carpeta con voces de Piper en español (.onnx y .onnx.json)")
    p.add_argument("--salida", type=Path, required=True, help="<output_dir>/<model_name> del entrenador")
    p.add_argument("--n-train", type=int, default=5000)
    p.add_argument("--n-test", type=int, default=500)
    p.add_argument("--grabaciones", type=Path, help="carpeta con una subcarpeta por persona")
    p.add_argument("--reservar", nargs="*", default=[], help="personas cuyas grabaciones se guardan para medir")
    p.add_argument("--repetir-reales", type=int, default=10)
    p.add_argument("--semilla", type=int, default=2026)
    p.add_argument("--espeak", type=Path, help="carpeta espeak-ng-data, si la del paquete tiene una ruta muy larga")
    p.add_argument("--rehacer", action="store_true", help="borra los clips que ya había")
    a = p.parse_args()

    azar = random.Random(a.semilla)
    if a.rehacer and a.salida.exists():
        for sub in ("positive_train", "positive_test", "negative_train", "negative_test"):
            shutil.rmtree(a.salida / sub, ignore_errors=True)
    voces = cargar_voces(a.voces, a.espeak)
    print("voces:", ", ".join(f"{n} ({h} hablante{'s' if h > 1 else ''})" for n, _, h in voces))
    llenar(a.salida / "positive_train", a.n_train, POSITIVAS, voces, azar)
    llenar(a.salida / "positive_test", a.n_test, POSITIVAS, voces, azar)
    llenar(a.salida / "negative_train", a.n_train, NEGATIVAS, voces, azar)
    llenar(a.salida / "negative_test", a.n_test, NEGATIVAS, voces, azar)
    if a.grabaciones:
        sumar_grabaciones(a.grabaciones, a.salida / "positive_train", a.salida / "positive_test", set(a.reservar), a.repetir_reales, azar)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile entrenar.py
"""Entrena «Hola Fructus» con el entrenador de openWakeWord 0.6.0, a partir de los clips que dejó generar_clips.py.

Usa solo los pasos «aumentar» y «entrenar» de openwakeword/train.py: los clips en español ya existen, así que no se genera nada
con su voz en inglés. Tres arreglos para que corra en un Python de hoy (el de Colab):
  1. compat_torchaudio: torchaudio ya no tiene `info` y `load` pide torchcodec.
  2. Un `generate_samples` de mentira: train.py lo importa siempre, aunque no se use, y el de verdad arrastra dependencias de inglés.
  3. Al final train.py convierte a TFLite con TensorFlow 2.8, que ya no se instala. El .onnx, que es lo que usa el navegador, ya
     quedó guardado antes; ese último paso se deja fallar y se avisa.

Uso:
    python entrenar.py --salida salida --rir mit_rirs --fondo audioset_16k fma_16k \
        --negativos openwakeword_features_ACAV100M_2000_hrs_16bit.npy --validacion validation_set_features.npy --pasos 10000
"""

import argparse
import logging
import multiprocessing
import runpy
import sys
from pathlib import Path

import yaml

MODELO = "hola_fructus"


def main() -> None:
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--salida", type=Path, required=True, help="carpeta donde generar_clips.py dejó <salida>/hola_fructus")
    p.add_argument("--rir", type=Path, nargs="+", required=True, help="respuestas de sala (.wav)")
    p.add_argument("--fondo", type=Path, nargs="+", required=True, help="ruido de fondo (.wav a 16 kHz)")
    p.add_argument("--negativos", type=Path, required=True, help="características de audio que no es la palabra (.npy)")
    p.add_argument("--validacion", type=Path, required=True, help="características para medir falsas activaciones (.npy)")
    p.add_argument("--pasos", type=int, default=10000)
    p.add_argument("--capa", type=int, default=32)
    p.add_argument("--peso-negativo", type=float, default=1500, help="peso máximo de los negativos (el de openWakeWord es 1500)")
    p.add_argument("--falsas-por-hora", type=float, default=0.2, help="meta del entrenador al elegir el mejor modelo")
    p.add_argument("--rehacer-caracteristicas", action="store_true", help="vuelve a aumentar los clips aunque ya estén")
    a = p.parse_args()
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")  # el avance del entrenador va por logging

    clips = a.salida / MODELO
    cuantos = {d: len(list((clips / d).glob("*.wav"))) for d in ("positive_train", "positive_test", "negative_train", "negative_test")}
    if min(cuantos.values()) == 0:
        raise SystemExit(f"Faltan clips en {clips}: {cuantos}. Corre primero generar_clips.py.")

    # El entrenador importa generate_samples siempre; con los clips ya hechos no se llama nunca.
    falso = a.salida / "_sin_piper_ingles"
    falso.mkdir(parents=True, exist_ok=True)
    (falso / "generate_samples.py").write_text(
        "def generate_samples(*args, **kwargs):\n    raise RuntimeError('Los clips se generan con generar_clips.py')\n"
    )

    config = {
        "model_name": MODELO,
        "target_phrase": ["hola fructus"],
        "custom_negative_phrases": [],
        # El entrenador solo genera si hay menos del 95 % de estos: se le dicen los que ya hay.
        "n_samples": cuantos["positive_train"],
        "n_samples_val": cuantos["positive_test"],
        "tts_batch_size": 50,
        "augmentation_batch_size": 16,
        "piper_sample_generator_path": str(falso),
        "output_dir": str(a.salida),
        "rir_paths": [str(r) for r in a.rir],
        "background_paths": [str(f) for f in a.fondo],
        "background_paths_duplication_rate": [1] * len(a.fondo),
        "false_positive_validation_data_path": str(a.validacion),
        "augmentation_rounds": 1,
        "feature_data_files": {"ACAV100M_sample": str(a.negativos)},
        "batch_n_per_class": {"ACAV100M_sample": 1024, "adversarial_negative": 50, "positive": 50},
        "model_type": "dnn",
        "layer_size": a.capa,
        "steps": a.pasos,
        "max_negative_weight": a.peso_negativo,
        "target_false_positives_per_hour": a.falsas_por_hora,
    }
    ruta_config = a.salida / f"{MODELO}.yml"
    ruta_config.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False))
    print(f"config: {ruta_config}")

    sys.path.insert(0, str(Path(__file__).parent))
    import compat_torchaudio

    compat_torchaudio.aplicar()

    import openwakeword.utils

    openwakeword.utils.download_models(model_names=["__solo_las_de_caracteristicas__"])  # melspectrogram y embedding

    # Los cargadores de datos del entrenador usan generadores: con «spawn» (macOS) no se pueden pasar a otro proceso.
    if sys.platform == "darwin":
        multiprocessing.set_start_method("fork", force=True)

    argumentos = ["--training_config", str(ruta_config), "--augment_clips", "--train_model"]
    if a.rehacer_caracteristicas:
        argumentos.append("--overwrite")
    sys.argv = ["train.py", *argumentos]
    onnx = a.salida / f"{MODELO}.onnx"
    try:
        runpy.run_module("openwakeword.train", run_name="__main__")
    except ImportError as e:
        if not onnx.exists():
            raise
        print(f"Aviso: no se pudo convertir a TFLite ({e.name}). No hace falta: el navegador usa el .onnx.")
    if not onnx.exists():
        raise SystemExit("El entrenamiento terminó sin dejar el .onnx.")
    print(f"Listo: {onnx} ({onnx.stat().st_size // 1024} KB)")


if __name__ == "__main__":
    main()

## 3. Ajustes

- `RESERVAR`: las personas cuyas grabaciones **no** se usan al entrenar. Quedan para medir, así el número final sale de voces que el
  modelo nunca oyó. Pon al menos una (idealmente el dueño).
- `PASOS`: 20,000 es un buen comienzo.

In [ ]:
RESERVAR = ["dueno"]
PASOS = 20000
CLIPS_ENTRENAR = 5000
CLIPS_VALIDAR = 500

## 4. Bajar voces, salas, ruido de fondo y características (~19 GB, unos 20 min)

In [ ]:
!python preparar_datos.py --destino . --fragmentos-fondo 2

## 5. Subir las grabaciones de «Hola Fructus» (`grabaciones.zip`)

Si todavía no hay grabaciones, salta esta celda: se entrena solo con voces sintéticas.

In [ ]:
from google.colab import files
subido = files.upload()
for nombre in subido:
    !unzip -q -o "{nombre}" -d .
!ls grabaciones

## 6. Generar los clips en español

In [ ]:
import os
reales = "--grabaciones grabaciones --reservar " + " ".join(RESERVAR) if os.path.isdir("grabaciones") else ""
!python generar_clips.py --voces voces --salida salida/hola_fructus --n-train {CLIPS_ENTRENAR} --n-test {CLIPS_VALIDAR} {reales}

## 7. Entrenar (30 a 60 min)

In [ ]:
!python entrenar.py --salida salida --rir mit_rirs --fondo fondo --negativos openwakeword_features_ACAV100M_2000_hrs_16bit.npy --validacion validation_set_features.npy --pasos {PASOS}

## 8. Bajar el modelo

Guarda `hola_fructus.onnx` y pásalo al proyecto: se mide con `medir.py`.

In [ ]:
from google.colab import files
files.download("salida/hola_fructus.onnx")